# Química de LAB004–012: theta_full frente a theta_defaults

Se reproducen los nueve experimentos históricos de **mosto natural** con parámetros congelados y se comparan las predicciones con las muestras reales normalizadas. **LAB004–012 participaron en la calibración: esto evalúa el ajuste sobre entrenamiento; no es un holdout ni una recalibración.**

El análisis incluye biomasa viable y muerta, nitrógeno, glucosa, fructosa, etanol y glicerol. G+F se presenta como variable derivada. No utiliza la capa de CO₂.

Todas las funciones específicas están en este SOURCE, que solo importa módulos científicos compartidos existentes. El EXECUTED se genera ejecutando el SOURCE completo. Las figuras quedan integradas; las tablas y la trazabilidad se guardan junto al notebook, sin figuras en results.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "fermentation_model/shared").is_dir())
FM = ROOT / "fermentation_model"
if str(FM) not in sys.path:
    sys.path.insert(0, str(FM))
from shared import run_new_must_glycerol_estimability_doe as kinetic
from laboratory_2026 import run_estimability_historical_by_medium as historical

NB_DIR = FM / "laboratory_2026/notebooks/calibration"
OUT = NB_DIR / "chemistry_lab004_012_theta_comparison_artifacts"
OUT.mkdir(parents=True, exist_ok=True)
NATURAL = FM / "laboratory_2026/results/estimability_historical_natural"
INPUTS = {
    "chemistry_normalized": historical.SECONDARY_DATA_PATH,
    "temperature_30min": NATURAL / "process_temperature_30min.csv",
    "theta_full": NATURAL / "theta_natural_full.csv",
    "theta_natural_11": NATURAL / "theta.csv",
    "theta_inherited_6": FM / "shared/results/new_must_glycerol_overnight_validation/theta_multistart_07.csv",
    "calibration_batch_summary": NATURAL / "batch_summary.csv",
    "calibration_support": NATURAL / "measurement_support.csv",
    "calibration_summary": NATURAL / "summary.json",
    "source_workbook": FM / "data/Laboratorio 2026/Vendimia_2026/mosto_natural_xthiol.xlsx",
    "source_calendar": FM / "data/Laboratorio 2026/raw_data/Fernanda Folch.ics",
    "prior_temporal_audit": FM / "docs/natural_must_temporal_audit_LAB001_018.md",
    "normalization_loader": FM / "shared/new_must_data_loader.py",
    "kinetic_model": Path(kinetic.__file__),
    "historical_loader": Path(historical.__file__),
}
assert all(p.is_file() for p in INPUTS.values())
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
INPUT_HASHES = {k: file_hash(p) for k,p in INPUTS.items()}
LABS = tuple(f"LAB{i:03d}" for i in range(4,13))
CORE = tuple(kinetic.STATE_NAMES)
STATES = ("X","Xd","N","G","F","GF","E","Gly")
LABELS = {"X":"Biomasa viable","Xd":"Biomasa muerta","N":"Nitrógeno asimilable (YAN)",
          "G":"Glucosa","F":"Fructosa","GF":"Glucosa + fructosa","E":"Etanol","Gly":"Glicerol"}
FACTORS = {s:1000. if s=="N" else 1. for s in STATES}
UNITS = {s:"mg N/L" if s=="N" else "g/L" for s in STATES}
COLORS = {"theta_full":"#0072B2","theta_defaults":"#D55E00"}
STYLES = {"theta_full":"-","theta_defaults":"--"}
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi":110,"font.size":10,"axes.titlesize":11})
pd.set_option("display.max_rows",170)
pd.set_option("display.max_columns",20)
print("Python:",sys.executable)
print("Salida:",OUT.relative_to(ROOT))

## 1. Vectores y procedencia

Los dos vectores comparten 11 parámetros reajustados con LAB004–012. Los otros seis de theta_full proceden de theta_multistart_07, ajustado con 19 experimentos: LAB004–012 naturales y MS007–016 sintéticos. Theta_defaults sustituye esos seis por los defaults **sin volver a ajustar los once restantes**. Se evalúa el efecto conjunto de esa sustitución, no dos optimizaciones independientes.

In [ ]:
parameter_table = pd.read_csv(INPUTS["theta_full"])
theta_full = parameter_table.set_index("parameter").theta.astype(float).to_dict()
DEFAULT_SIX = {"sN":18.,"qXG":.1125,"qXF":.1125,"sG":.03,"sF":.03,"m0":.01}
assert set(theta_full)==set(kinetic.FULL17)
assert all(np.isclose(kinetic.DEFAULT_THETA[k],v) for k,v in DEFAULT_SIX.items())
theta_defaults = {**theta_full,**DEFAULT_SIX}
THETAS = {"theta_full":theta_full,"theta_defaults":theta_defaults}
assert {k for k in theta_full if theta_full[k]!=theta_defaults[k]}==set(DEFAULT_SIX)
natural11 = pd.read_csv(INPUTS["theta_natural_11"]).set_index("parameter")
inherited = pd.read_csv(INPUTS["theta_inherited_6"],index_col=0).iloc[:,0]
for name in kinetic.FULL17:
    reference = inherited[name] if name in DEFAULT_SIX else natural11.loc[name,"theta"]
    assert np.isclose(theta_full[name],reference),name
parameter_table["theta_full"] = parameter_table.parameter.map(theta_full)
parameter_table["theta_defaults"] = parameter_table.parameter.map(theta_defaults)
parameter_table["changes"] = parameter_table.parameter.isin(DEFAULT_SIX)
parameter_table["origin"] = np.where(parameter_table.changes,
    "Heredado del ajuste mixto / sustituido por default",
    "Reajustado con LAB004–012; idéntico en ambos vectores")
display(parameter_table[["parameter","theta_full","theta_defaults","origin"]])

## 2. Datos y condiciones de entrada

Se utiliza la misma tabla normalizada de la calibración histórica, secondary_metabolite_long.csv, filtrada a LAB004–012 naturales. No se interpolan observaciones ni se eliminan según el error. Biomasa y etanol conservan las conversiones de unidades históricas. La auditoría muestra archivos de origen, muestras disponibles y posibles valores afectados por las cotas heredadas.

Las condiciones iniciales son la primera medición finita de cada especie, según el cargador histórico; se identifica si proviene de una muestra posterior al inicio. Se mantienen iguales en ambos vectores.

La temperatura usa el archivo archivado cada 30 minutos y la misma representación histórica: malla del modelo de 2 h más tiempos de muestras y cambios de consigna. Fuera de cobertura del sensor se conserva la temperatura nominal. No se cambia ni suaviza nuevamente.

Los pulsos de N se aplican completos como saltos. Una muestra coincidente con un pulso representa el estado **anterior** a la adición. No se excluye la respuesta a la nutrición.

In [ ]:
data = historical.load_historical_medium_data("natural")
data = data[data.batch.isin(LABS)].copy()
assert tuple(sorted(data.batch.unique()))==LABS
assert not data.duplicated(["batch","time_h"]).any()
temperature = pd.read_csv(INPUTS["temperature_30min"])
batches,temperature_alignment = historical.augment_natural_batches_with_temperature(
    kinetic.make_batches(data),temperature)
assert tuple(b.batch for b in batches)==LABS
batch_by_lab = {b.batch:b for b in batches}
initial_rows,audit_rows,observation_rows = [],[],[]
for batch in batches:
    raw = data[data.batch.eq(batch.batch)].sort_values("time_h")
    sensor = temperature[temperature.batch.eq(batch.batch)]
    assert not sensor.empty
    for state in CORE:
        column = kinetic.STATE_COLUMNS[state]
        finite = raw.loc[raw[column].notna(),["time_h",column]]
        first_time = float(finite.time_h.iloc[0]) if len(finite) else np.nan
        initial_rows.append({"batch":batch.batch,"state":state,"unit":UNITS[state],
            "initial_value":batch.initials[state]*FACTORS[state],"initial_source_time_h":first_time,
            "source_after_model_start":bool(first_time>batch.time[0]+1e-8) if len(finite) else False,
            "initial_uses_fallback":not len(finite),"n_observations":len(finite),
            "n_negative_source":int((finite[column]<0).sum()),
            "n_values_changed_by_model_bounds":int(np.count_nonzero(
                finite[column].to_numpy()!=np.maximum(finite[column].to_numpy(),kinetic.STATE_BOUNDS[state][0])))})
        for t,val in zip(batch.time,batch.observations[state]):
            if np.isfinite(val):
                observation_rows.append({"batch":batch.batch,"state":state,"time_h":float(t),
                    "observed":float(val)*FACTORS[state],"unit":UNITS[state],
                    "initial_input_anchor":bool(np.isclose(t,first_time,atol=1e-8,rtol=0))})
    audit_rows.append({"batch":batch.batch,"n_source_rows":len(raw),"horizon_h":float(batch.time[-1]),
        "T_min_C":batch.temperature_c.min(),"T_max_C":batch.temperature_c.max(),
        "T_sensor_start_h":sensor.t_h.min(),"T_sensor_end_h":sensor.t_h.max(),
        "n_model_times_outside_sensor":int(((batch.time<sensor.t_h.min())|(batch.time>sensor.t_h.max())).sum()),
        "N_pulses_h_mg_L":"; ".join(f"{t:g} h: {dose*1000:g} mg/L" for t,dose in batch.pulses["N"]),
        "source_files":"; ".join(sorted(raw.source_file.dropna().unique()))})
initial_audit = pd.DataFrame(initial_rows)
batch_audit = pd.DataFrame(audit_rows)
observations = pd.DataFrame(observation_rows)
counts = observations.groupby("state").size()
support = pd.read_csv(INPUTS["calibration_support"]).set_index("state")
reference_batches = pd.read_csv(INPUTS["calibration_batch_summary"]).set_index("batch")
for state in CORE:
    assert counts[state]==support.loc[state,"n_observations"]
for batch in batches:
    assert np.isclose(batch.time[-1],reference_batches.loc[batch.batch,"t_max_h"])
    for state in CORE:
        assert np.isclose(batch.initials[state],reference_batches.loc[batch.batch,f"{state}0"])
display(batch_audit.round(3))
display(initial_audit.pivot(index="batch",columns="state",values="initial_value").reindex(columns=CORE).round(4))
display(counts.rename("n_muestras_reales").to_frame())
late_initials = initial_audit[initial_audit.source_after_model_start]
display(Markdown(f"**{len(late_initials)}** condiciones iniciales proceden de una muestra posterior al inicio. "
    "Se mantiene y documenta esta convención; no constituye un pronóstico independiente de esas entradas."))
if len(late_initials):
    display(late_initials)
print("Valores negativos originales:",int(initial_audit.n_negative_source.sum()))
print("Observaciones afectadas por cotas:",int(initial_audit.n_values_changed_by_model_bounds.sum()))

In [ ]:
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,batch in zip(axes.flat,batches):
    sensor = temperature[temperature.batch.eq(batch.batch)]
    ax.plot(sensor.t_h,sensor["T"],color="grey",lw=.8,alpha=.5,label="Sensor archivado · 30 min")
    ax.plot(batch.time,batch.temperature_c,color="#6A3D9A",lw=1.5,label="Temperatura usada por el modelo")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",ls=":",lw=1.2)
    ax.set(title=batch.batch,xlabel="Tiempo histórico [h]",ylabel="Temperatura [°C]",xlim=(0,batch.time[-1]))
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=2)
fig.suptitle("Temperatura de entrada · líneas rosadas: pulsos de N",y=.995)
plt.show()
plt.close(fig)

## 3. Simulación y métricas

Se integra el mismo modelo de siete estados sin ajustar parámetros, ganancias ni tiempos. El error se calcula sobre una simulación en los tiempos originales del ajuste. Una segunda malla de 0,25 h sirve para dibujar curvas y se verifica su concordancia numérica en los tiempos compartidos.

El error es **predicción − observación**. RMSE, MAE y sesgo tienen unidades g/L, excepto N (mg N/L). R² queda indefinido con varianza observada cero. NRMSE divide RMSE por el rango observado del grupo evaluado; no se calcula para rango cero.

El criterio principal excluye la primera observación de cada especie usada como condición inicial, aunque sea posterior a cero. Esas muestras siguen visibles como círculos abiertos. Se exporta además una tabla que incluye todas las observaciones, según el objetivo histórico.

**WRMSE = raíz de la media de (error / sigma_histórica)².** Las sigmas son las ponderaciones históricas del ajuste, no incertidumbres analíticas certificadas. Permiten comparar especies sin mezclar errores físicos de distinta escala.

G+F solo se calcula con muestras simultáneas de G y F. No tiene una sigma inventada ni se cuenta otra vez en el resumen ponderado.

In [ ]:
point_frames,trajectory_frames,numerical_rows = [],[],[]
for label,theta in THETAS.items():
    for batch in batches:
        scored = kinetic.simulate(batch,theta,batch.time,sample_event_order="sample_before_action")
        assert scored is not None and np.isfinite(scored.to_numpy()).all(),(label,batch.batch)
        events = [t for t,_ in batch.pulses["N"]]
        post_events = [t+1e-4 for t in events if t+1e-4<batch.time[-1]]
        grid = np.unique(np.r_[batch.time,np.arange(batch.time[0],batch.time[-1],.25),events,post_events])
        dense = kinetic.simulate(batch,theta,grid,sample_event_order="sample_before_action")
        assert dense is not None and np.isfinite(dense.to_numpy()).all()
        dense["GF"] = dense.G+dense.F
        for state in CORE:
            vals = np.asarray(batch.observations[state],float)
            mask = np.isfinite(vals)
            sig = kinetic.sigma_for_state(state,vals[mask])
            delta = dense.loc[batch.time,state].to_numpy()[mask]-scored[state].to_numpy()[mask]
            numerical_rows.append({"theta":label,"batch":batch.batch,"state":state,"unit":UNITS[state],
                "max_abs_dense_minus_scored":np.max(np.abs(delta))*FACTORS[state],
                "max_delta_in_historical_sigma":float(np.max(np.abs(delta)/sig))})
            p = observations[observations.batch.eq(batch.batch)&observations.state.eq(state)].copy()
            p["predicted"] = scored.loc[p.time_h.to_numpy(),state].to_numpy()*FACTORS[state]
            p["sigma_historical"] = sig*FACTORS[state]
            p["theta"] = label
            point_frames.append(p)
        paired = observations[observations.batch.eq(batch.batch)&observations.state.eq("G")].merge(
            observations[observations.batch.eq(batch.batch)&observations.state.eq("F")],
            on=["batch","time_h"],suffixes=("_G","_F"),validate="one_to_one")
        point_frames.append(pd.DataFrame({"batch":batch.batch,"state":"GF","time_h":paired.time_h,
            "observed":paired.observed_G+paired.observed_F,
            "predicted":scored.loc[paired.time_h.to_numpy(),["G","F"]].sum(axis=1).to_numpy(),
            "unit":"g/L","initial_input_anchor":paired.initial_input_anchor_G|paired.initial_input_anchor_F,
            "sigma_historical":np.nan,"theta":label}))
        for state in STATES:
            trajectory_frames.append(pd.DataFrame({"theta":label,"batch":batch.batch,"state":state,
                "time_h":dense.index.to_numpy(),"predicted":dense[state].to_numpy()*FACTORS[state],"unit":UNITS[state]}))
points = pd.concat(point_frames,ignore_index=True)
points["error"] = points.predicted-points.observed
points["used_in_primary_score"] = ~points.initial_input_anchor
trajectories = pd.concat(trajectory_frames,ignore_index=True)
numerical_check = pd.DataFrame(numerical_rows)
assert not points.duplicated(["theta","batch","state","time_h"]).any()
for state in STATES:
    a = points[points.theta.eq("theta_full")&points.state.eq(state)].sort_values(["batch","time_h"])
    b = points[points.theta.eq("theta_defaults")&points.state.eq(state)].sort_values(["batch","time_h"])
    assert np.array_equal(a.observed.to_numpy(),b.observed.to_numpy())
    assert np.array_equal(a.initial_input_anchor.to_numpy(),b.initial_input_anchor.to_numpy())
print("Combinaciones LAB/vector:",len(THETAS)*len(batches),"; dos mallas por combinación")
print("Máxima discrepancia entre mallas / sigma:",numerical_check.max_delta_in_historical_sigma.max())
assert numerical_check.max_delta_in_historical_sigma.max()<.05,"Revisar concordancia de mallas"

In [ ]:
def summarize_errors(frame,group_columns,exclude_anchors=True):
    selected = frame[~frame.initial_input_anchor] if exclude_anchors else frame
    rows = []
    for keys,g in selected.groupby(group_columns,sort=True):
        keys = keys if isinstance(keys,tuple) else (keys,)
        error,obs,sigma = (g[c].to_numpy(float) for c in ["error","observed","sigma_historical"])
        total = np.sum((obs-obs.mean())**2)
        rms = np.sqrt(np.mean(error**2))
        rows.append({**dict(zip(group_columns,keys)),"n":len(g),"unit":g.unit.iloc[0],
            "RMSE":rms,"MAE":np.mean(np.abs(error)),"bias_pred_minus_obs":error.mean(),
            "R2":1-np.sum(error**2)/total if total>1e-16 else np.nan,
            "NRMSE_range":rms/np.ptp(obs) if np.ptp(obs)>1e-12 else np.nan,
            "WRMSE_historical":np.sqrt(np.mean((error/sigma)**2)) if np.isfinite(sigma).all() else np.nan})
    return pd.DataFrame(rows)

metrics = summarize_errors(points,["theta","batch","state"])
metrics_all = summarize_errors(points,["theta","batch","state"],exclude_anchors=False)
by_variable = summarize_errors(points,["theta","state"])
weighted_rows = []
for label,p in points[points.used_in_primary_score&points.state.isin(CORE)].groupby("theta"):
    for lab in [*LABS,"TODOS"]:
        g = p if lab=="TODOS" else p[p.batch.eq(lab)]
        per_group = metrics[metrics.theta.eq(label)&metrics.state.isin(CORE)]
        if lab!="TODOS":
            per_group = per_group[per_group.batch.eq(lab)]
        weighted_rows.append({"theta":label,"batch":lab,"n":len(g),
            "WRMSE_historical_pooled":np.sqrt(np.mean((g.error/g.sigma_historical)**2)),
            "mean_WRMSE_equal_lab_state":per_group.WRMSE_historical.mean()})
weighted_summary = pd.DataFrame(weighted_rows)
display(Markdown("**Errores por variable, agrupando las muestras evaluadas de los nueve LAB.**"))
display(by_variable.sort_values(["state","theta"]).round(4))
display(Markdown("**Errores por experimento: siete especies, sin contar G+F dos veces.**"))
display(weighted_summary.round(4))

## 4. Curvas por variable y experimento

Los puntos negros representan todas las muestras disponibles; los círculos abiertos son entradas iniciales excluidas del error principal. Las líneas rosadas marcan nutrición. Los títulos indican el RMSE sin entradas iniciales. No se reajusta el reloj para mejorar el ajuste.

Las curvas se muestran por variable, con un panel por cada LAB, para identificar diferencias de velocidad, amplitud, agotamiento y respuesta a las adiciones.

In [ ]:
state = "X"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "Xd"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "N"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    # Margen visual bajo cero: las concentraciones no se modifican.
    ax.set_ylim(bottom=-0.02 * ax.get_ylim()[1])
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "G"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "F"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "GF"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "E"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

In [ ]:
state = "Gly"
fig,axes = plt.subplots(3,3,figsize=(15,10),layout="constrained")
for ax,lab in zip(axes.flat,LABS):
    batch = batch_by_lab[lab]
    p = points[points.theta.eq("theta_full")&points.batch.eq(lab)&points.state.eq(state)]
    for label in THETAS:
        curve = trajectories[trajectories.theta.eq(label)&trajectories.batch.eq(lab)&trajectories.state.eq(state)]
        ax.plot(curve.time_h,curve.predicted,color=COLORS[label],ls=STYLES[label],lw=1.8,label=label)
    measured,anchor = p[~p.initial_input_anchor],p[p.initial_input_anchor]
    ax.scatter(measured.time_h,measured.observed,color="black",s=23,zorder=5,label="Muestras reales")
    ax.scatter(anchor.time_h,anchor.observed,facecolors="white",edgecolors="black",s=35,zorder=6,label="Entrada inicial")
    for t,_ in batch.pulses["N"]:
        ax.axvline(t,color="#CC79A7",lw=1,ls=":",alpha=.85)
    m = metrics[metrics.batch.eq(lab)&metrics.state.eq(state)].set_index("theta")
    ax.set(title=f"{lab} · RMSE full={m.loc['theta_full','RMSE']:.3g}; defaults={m.loc['theta_defaults','RMSE']:.3g}",
           xlabel="Tiempo histórico [h]",ylabel=f"{state} [{UNITS[state]}]",xlim=(0,batch.time[-1]))
    ax.set_ylim(bottom=0)
fig.get_layout_engine().set(rect=(0,0,1,.90))
handles,labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=4)
fig.suptitle(f"{LABELS[state]} · LAB004–012 · RMSE sin entradas iniciales",y=.995)
plt.show()
plt.close(fig)

## 5. Errores por especie y experimento

Se muestran RMSE, MAE, sesgo y R² en cada par LAB/especie. Sesgo positivo significa sobreestimación.

Las dos matrices de WRMSE usan una misma escala de color; 1 equivale a una escala histórica de error cuadrático medio, sin implicar un umbral estadístico de aceptación. En la matriz de diferencias de RMSE, **azul favorece full y rojo favorece defaults**. Las diferencias pequeñas no prueban superioridad estadística.

In [ ]:
for state in STATES:
    display(Markdown(f"**{LABELS[state]} — {UNITS[state]}**"))
    display(metrics[metrics.state.eq(state)].pivot(index="batch",columns="theta",
        values=["n","RMSE","MAE","bias_pred_minus_obs","R2"]).round(4))

fig,axes = plt.subplots(1,2,figsize=(15,6),layout="constrained")
vmax = metrics[metrics.state.isin(CORE)].WRMSE_historical.max()
for ax,label in zip(axes,THETAS):
    grid = metrics[metrics.theta.eq(label)].pivot(index="batch",columns="state",values="WRMSE_historical").reindex(index=LABS,columns=CORE)
    ax.grid(False)
    im = ax.imshow(grid,vmin=0,vmax=vmax,cmap="YlOrRd",aspect="auto")
    for i in range(len(LABS)):
        for j in range(len(CORE)):
            value = grid.iloc[i,j]
            ax.text(j,i,f"{value:.1f}",ha="center",va="center",color="white" if value>.6*vmax else "black",fontsize=9)
    ax.set(xticks=range(len(CORE)),xticklabels=CORE,yticks=range(len(LABS)),yticklabels=LABS,title=label)
fig.colorbar(im,ax=axes,label="WRMSE / escala histórica")
fig.suptitle("Error de las siete especies originales · sin entradas iniciales")
plt.show()
plt.close(fig)

In [ ]:
comparison = metrics.pivot(index=["batch","state"],columns="theta",values="RMSE").reset_index()
comparison["delta_RMSE_full_minus_defaults"] = comparison.theta_full-comparison.theta_defaults
comparison["relative_change_percent"] = np.where(comparison.theta_defaults>1e-12,
    100*comparison.delta_RMSE_full_minus_defaults/comparison.theta_defaults,np.nan)
delta = comparison.pivot(index="batch",columns="state",values="relative_change_percent").reindex(index=LABS,columns=STATES)
limit = max(1.,float(np.nanmax(np.abs(delta.to_numpy()))))
fig,ax = plt.subplots(figsize=(12,6),layout="constrained")
ax.grid(False)
im = ax.imshow(delta,cmap="RdBu_r",vmin=-limit,vmax=limit,aspect="auto")
for i in range(len(LABS)):
    for j in range(len(STATES)):
        ax.text(j,i,f"{delta.iloc[i,j]:+.0f}%",ha="center",va="center",fontsize=9,
            color="white" if abs(delta.iloc[i,j])>.65*limit else "black")
ax.set(xticks=range(len(STATES)),xticklabels=STATES,yticks=range(len(LABS)),yticklabels=LABS,
       title="Cambio de RMSE al usar full frente a defaults · negativo favorece full")
fig.colorbar(im,ax=ax,label="100 × (RMSE full − RMSE defaults) / RMSE defaults")
plt.show()
plt.close(fig)

In [ ]:
fig,axes = plt.subplots(2,4,figsize=(16,7),layout="constrained")
for ax,state in zip(axes.flat,STATES):
    m = by_variable[by_variable.state.eq(state)].set_index("theta").reindex(THETAS)
    ax.bar(range(2),m.RMSE,color=[COLORS[label] for label in THETAS],width=.6)
    ax.set(xticks=range(2),xticklabels=["full","defaults"],ylabel=f"RMSE [{UNITS[state]}]",title=LABELS[state])
fig.suptitle("RMSE por variable · todas las muestras evaluadas de LAB004–012 · sin entradas iniciales")
plt.show()
plt.close(fig)

## 6. Conclusiones calculadas y límites de interpretación

Las conclusiones se generan desde los errores de esta ejecución. El resumen por observación da más peso a especies y experimentos con más muestras; por eso se muestra también un promedio con igual peso por par LAB/especie. No se reajusta ni selecciona un nuevo vector con estos errores.

In [ ]:
full_score = weighted_summary.query("theta == 'theta_full' and batch == 'TODOS'").iloc[0]
def_score = weighted_summary.query("theta == 'theta_defaults' and batch == 'TODOS'").iloc[0]
core_comparison = comparison[comparison.state.isin(CORE)]
wins_full = int((core_comparison.delta_RMSE_full_minus_defaults < -1e-10).sum())
wins_defaults = int((core_comparison.delta_RMSE_full_minus_defaults > 1e-10).sum())
ties = len(core_comparison)-wins_full-wins_defaults
n_scored = len(points[points.theta.eq("theta_full")&points.used_in_primary_score&points.state.isin(CORE)])
regional_rows = []
for region, labs in {"LAB004–009": LABS[:6], "LAB010–012": LABS[6:]}.items():
    for label in THETAS:
        p = points[points.theta.eq(label) & points.batch.isin(labs) & points.used_in_primary_score & points.state.isin(CORE)]
        regional_rows.append({"region": region, "theta": label, "n": len(p),
            "WRMSE_historical": float(np.sqrt(np.mean((p.error/p.sigma_historical)**2)))})
regional_summary = pd.DataFrame(regional_rows)
regional_pivot = regional_summary.pivot(index="region", columns="theta", values="WRMSE_historical")
nitrogen_conflicts = points[points.theta.eq("theta_full") & points.state.eq("N") & points.used_in_primary_score
    & points.predicted.le(5) & points.observed.gt(20)]
lines = [
    f"- Se evaluaron **9 experimentos**, **{len(observations)} observaciones originales** de siete especies "
    f"y **{n_scored}** observaciones sin entradas iniciales. G+F se analiza aparte.",
    f"- WRMSE global por observación: **full = {full_score.WRMSE_historical_pooled:.3f}**, "
    f"**defaults = {def_score.WRMSE_historical_pooled:.3f}**. Menor es mejor según las ponderaciones históricas.",
    f"- Promedio de WRMSE con igual peso por par LAB/especie: "
    f"**full = {full_score.mean_WRMSE_equal_lab_state:.3f}**, **defaults = {def_score.mean_WRMSE_equal_lab_state:.3f}**.",
    f"- De **63 pares LAB/especie originales**, full tiene menor RMSE en **{wins_full}**, "
    f"defaults en **{wins_defaults}**; hay **{ties}** empates numéricos. No es una prueba de significancia.",
]
variable_comparison = by_variable.pivot(index="state",columns="theta",values="RMSE")
for state in STATES:
    a,b = variable_comparison.loc[state,["theta_full","theta_defaults"]]
    winner = "theta_full" if a<b else "theta_defaults" if b<a else "empate"
    lines.append(f"- **{LABELS[state]}:** RMSE full **{a:.4g}** frente a defaults **{b:.4g} {UNITS[state]}**; menor: **{winner}**.")
worst = metrics[metrics.state.isin(CORE)].sort_values("WRMSE_historical",ascending=False).head(6)
lines += [
    "- Los mayores errores normalizados aparecen debajo. Una buena curva de N por sí sola no acredita un buen ajuste de azúcares, etanol o biomasa.",
    "- LAB004–012 participaron en la calibración. Los resultados describen ajuste histórico; la transferencia requiere experimentos independientes.",
    "- Defaults mantiene once parámetros optimizados con los otros seis fijados a sus valores full: esta comparación no equivale a dos calibraciones igualmente optimizadas.",
    f"- Se conservaron pulsos, temperatura y condiciones iniciales históricos. Hay **{len(late_initials)}** "
    f"entradas iniciales posteriores al inicio y **{int(initial_audit.n_values_changed_by_model_bounds.sum())}** "
    "observaciones afectadas por cotas heredadas; su auditoría debe acompañar la interpretación.",
]
for region, row in regional_pivot.iterrows():
    lines.append(f"- **{region}:** WRMSE full **{row['theta_full']:.3f}**, defaults **{row['theta_defaults']:.3f}**. "
        "La preferencia entre vectores cambia entre estos grupos; el promedio global no representa por igual todos los experimentos.")
lines.append(f"- **Nitrógeno:** con full hay **{len(nitrogen_conflicts)} muestras de {nitrogen_conflicts.batch.nunique()} LAB** "
    "con predicción ≤ 5 mg N/L y observación > 20 mg N/L. El agotamiento predicho no reproduce todo el nitrógeno residual medido.")
lines.append("- En las curvas de LAB010–012 se observan consumos de azúcares y aumentos de etanol anteriores a las muestras intermedias. "
    "Este diagnóstico documenta la discrepancia; no permite atribuirla por sí solo a la temperatura, a parámetros o al registro de entradas.")
display(Markdown("\n".join(lines)))
display(regional_summary.round(4))
display(worst[["theta","batch","state","n","unit","RMSE","bias_pred_minus_obs","WRMSE_historical"]].round(4))
conclusions = {"lines":lines,"n_original_observations":len(observations),"n_scored_core_per_theta":n_scored,
    "full_better_core_pairs":wins_full,"defaults_better_core_pairs":wins_defaults,"ties_core_pairs":ties,
    "is_holdout":False,"recalibration_performed":False}

In [ ]:
assert INPUT_HASHES=={k:file_hash(p) for k,p in INPUTS.items()},"Se modificó una entrada"
assert len(metrics)==2*9*8
assert len(numerical_check)==2*9*7
assert set(points.batch)==set(LABS)
for label in THETAS:
    assert len(points[points.theta.eq(label)&points.state.isin(CORE)])==len(observations)
for row in metrics.itertuples():
    g = points[points.theta.eq(row.theta)&points.batch.eq(row.batch)&points.state.eq(row.state)&points.used_in_primary_score]
    assert len(g)==row.n
    assert np.isclose(row.RMSE**2,np.dot(g.error,g.error)/len(g))
    assert np.isclose(row.MAE,np.abs(g.error).mean())
    assert np.isclose(row.bias_pred_minus_obs,g.error.mean())
tables = {
    "parameter_comparison.csv":parameter_table,
    "batch_inputs.csv":batch_audit,
    "initial_conditions_audit.csv":initial_audit,
    "temperature_alignment.csv":temperature_alignment,
    "observations_normalized.csv":data,
    "observation_predictions.csv":points,
    "chemistry_trajectories.csv":trajectories,
    "metrics_by_lab_variable.csv":metrics,
    "metrics_by_lab_variable_with_initials.csv":metrics_all,
    "metrics_by_variable.csv":by_variable,
    "weighted_summary_by_lab.csv":weighted_summary,
    "rmse_comparison.csv":comparison,
    "numerical_grid_check.csv":numerical_check,
}
for name,table in tables.items():
    table.to_csv(OUT/name,index=False)
manifest = {
    "created_at":datetime.now(ZoneInfo("America/Santiago")).isoformat(),
    "source_notebook":str((NB_DIR/"chemistry_lab004_012_theta_comparison.source.ipynb").relative_to(ROOT)),
    "experiments":list(LABS),"is_holdout":False,"fit_performed":False,"co2_layer_used":False,
    "initial_anchors_excluded_from_primary_errors":True,"original_states":list(CORE),
    "derived_state_excluded_from_weighted_summary":"GF",
    "sigma_definition":"historical model weighting; not certified analytical uncertainty",
    "temperature_policy":"archived 30-min sensor; historical 2h model grid; nominal outside sensor coverage",
    "pulse_order":"sample_before_action","dense_plot_step_h":.25,"python":sys.executable,
    "versions":{"numpy":np.__version__,"pandas":pd.__version__},
    "inputs":{k:{"path":str(p.relative_to(ROOT)),"sha256":INPUT_HASHES[k]} for k,p in INPUTS.items()},
    "tables":list(tables),"figures_saved_to_disk":False,
}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
(OUT/"conclusions.json").write_text(json.dumps(conclusions,indent=2,ensure_ascii=False),encoding="utf-8")
print(f"Verificado: {len(batches)} LAB, {len(THETAS)} vectores, {len(metrics)} métricas LAB/especie, {len(tables)} tablas.")
print("Entradas intactas. Figuras integradas en EXECUTED; sin archivos de figuras en results.")

## 7. Revisión documental de LAB006 y LAB010–012

Se revisan en modo de solo lectura la [planilla original](../../../data/Laboratorio%202026/Vendimia_2026/mosto_natural_xthiol.xlsx), el [calendario operativo](../../../data/Laboratorio%202026/raw_data/Fernanda%20Folch.ics), el cargador compartido y la [auditoría temporal previa](../../../docs/natural_must_temporal_audit_LAB001_018.md). Los registros se identifican por código de ensayo, no solo por un nombre de archivo que pueda confundirse con los DOE sintéticos.

**Hallazgos documentados:** LAB006 está asignado al protocolo térmico C (16→18→21 °C); LAB010 al D (15 °C isotérmico), LAB011 al E (18→15→21 °C) y LAB012 al F (15→18→21 °C). Los perfiles medidos ya se usan en las simulaciones. Abril registra Springferm Xtrem + FDA; mayo registra Viniliquid, 4 mL y un aporte declarado de 80 ppm YAN.

Los eventos de siembra registran **1 g de Zymaflore X5 para 2 L en los cuatro experimentos**. Las fuentes revisadas no explican explícitamente por qué las concentraciones celulares iniciales difieren tanto, ni documentan una dosis distinta de inóculo o una corrección de dilución que justifique cambiar X₀. La diferencia térmica y de nutrición sí está documentada; no demuestra por sí sola un error de biomasa. Los escenarios siguientes se realizan como diagnóstico incluso existiendo esas diferencias de protocolo.

**Salvedades importantes:** los eventos de mayo figuran como planificados. LAB010 dice «sin pulso automático» en Maestro, mientras Eventos y calendario contienen un segundo pulso planificado; eso no permite afirmar que no hubiera adición manual. Las horas tampoco coinciden entre todas las fuentes. Las etiquetas «Pre-inóculo», «Planificado» o «No realizado» del muestreo no se usan para borrar resultados analíticos existentes ni certifican el estado de ejecución.

La biomasa empleada por el modelo procede del recuento viable Oculyze convertido con **30 pg/célula**: un millón de células/mL equivale a 0,03 g/L. No se interpreta «Viability» como un porcentaje. La auditoría siguiente verifica esa transformación contra las celdas originales y los datos usados por el modelo.

In [ ]:
from openpyxl import load_workbook
from dataclasses import replace
from datetime import timezone
import re
from shared import new_must_data_loader as sx_loader

SX_LABS = ("LAB006", "LAB010", "LAB011", "LAB012")
def sx_sheet_records(sheet):
    header = [c.value for c in next(sheet.iter_rows())]
    records = []
    for row in sheet.iter_rows(min_row=2):
        record = {header[i]: c.value for i,c in enumerate(row) if i < len(header) and header[i] is not None}
        record["_row"] = row[0].row
        records.append(record)
    return records

sx_book = load_workbook(INPUTS["source_workbook"], read_only=True, data_only=True)
sx_master = sx_sheet_records(sx_book["Maestro_ensayos"])
sx_events = sx_sheet_records(sx_book["Eventos_operacion"])
sx_primary = sx_sheet_records(sx_book["Resultados_primarios"])
sx_homologated = sx_sheet_records(sx_book["Datos_homologados"])
sx_protocol_rows, sx_trace_rows = [], []
for lab in SX_LABS:
    code = "ING26-" + lab
    master = next(r for r in sx_master if r.get("Código ensayo") == code)
    events = [r for r in sx_events if (r.get("Código ensayo imputado") or r.get("Código ensayo")) == code]
    seed = next(r for r in events if r.get("Tipo evento") == "Siembra")
    first = next(sx_book[lab].iter_rows(min_row=2, max_row=2))
    first_values = {c.column_letter: c.value for c in first}
    primary = next(r for r in sx_primary if r.get("Código muestra") == first_values["A"])
    homologated = next(r for r in sx_homologated if r.get("ID") == first_values["A"])
    initial_nut = next(r for r in events if r.get("Tipo evento") == "Pulso nutricional 1")
    n_rows = data[data.batch.eq(lab)].N_kg_m3.notna().sum()
    e_rows = data[data.batch.eq(lab)].E_g_l.notna().sum()
    x_from_count = float(first_values["F"]) * sx_loader.MILLION_CELLS_ML_TO_KG_M3
    assert np.isclose(x_from_count, batch_by_lab[lab].initials["X"])
    assert np.isclose(float(primary["Oculyze Viability"]), float(first_values["F"]))
    sx_protocol_rows.append({"LAB":lab, "condicion":master["Condición"], "protocolo_registrado":master["Observaciones"],
        "inoculo_registrado":f"{seed['Dosis']} {seed['Unidad']}", "volumen_L":master["Volumen inicial (L)"],
        "nutriente":initial_nut["Producto/Nutriente"], "dosis_nutriente":f"{initial_nut['Dosis']} {initial_nut['Unidad']}",
        "estado_evento_siembra":seed.get("Estado") or "No consignado", "X0_g_L":x_from_count,
        "muestras_N_incluye_inicial":int(n_rows), "muestras_E_incluye_inicial":int(e_rows),
        "fuente_protocolo":f"Maestro_ensayos!F{master['_row']}:P{master['_row']}",
        "fuente_siembra":f"Eventos_operacion!B{seed['_row']}:O{seed['_row']}"})
    sx_trace_rows.append({"LAB":lab, "recuento_viable_millones_mL":first_values["F"],
        "X0_modelo_g_L":batch_by_lab[lab].initials["X"], "primera_muestra_fecha_hora":first_values["B"],
        "fuente_recuento":f"{lab}!F2; Resultados_primarios!H{primary['_row']}",
        "t_hoja_LAB_h":first_values["C"], "t_Datos_homologados_h":homologated["t"],
        "E0_hoja_porcentaje_vv":first_values["Q"],
        "Alcolyzer_primario_real":primary.get("Cf Alcolyzer Real (% v/v)"),
        "fuente_etanol":f"{lab}!Q2; Resultados_primarios!T{primary['_row']}",
        "hito_muestreo":first_values.get("AU"), "estado_muestreo":first_values.get("AZ")})
sx_book.close()
sx_protocol_audit = pd.DataFrame(sx_protocol_rows)
sx_initial_trace = pd.DataFrame(sx_trace_rows)
display(sx_protocol_audit)
display(sx_initial_trace)

# El calendario es una segunda referencia de horarios, no confirmación de ejecución.
sx_calendar = re.sub(r"\r?\n[ \t]", "", INPUTS["source_calendar"].read_text(encoding="utf-8-sig"))
sx_calendar_rows = []
for block in sx_calendar.split("BEGIN:VEVENT")[1:]:
    block = block.split("END:VEVENT")[0]
    values = dict(line.split(":",1) for line in block.splitlines() if ":" in line)
    summary = values.get("SUMMARY", "")
    lab_match = re.search(r"LAB(?:006|010|011|012)", block)
    if not lab_match or not any(k in summary for k in ["Siembra", "Pulso nutricional"]):
        continue
    stamp = datetime.strptime(values["DTSTART"], "%Y%m%dT%H%M%SZ").replace(tzinfo=timezone.utc)
    sx_calendar_rows.append({"LAB":lab_match.group(), "evento":summary,
        "hora_calendario_local":stamp.astimezone(ZoneInfo("America/Santiago")).isoformat()})
sx_calendar_audit = pd.DataFrame(sx_calendar_rows).sort_values(["LAB","hora_calendario_local"])
display(sx_calendar_audit)

### 7.1 Qué parte de la documentación sigue sin resolver

- **Inóculo efectivo:** no se encontró una explicación explícita de las diferencias de recuento inicial en las fuentes revisadas. Una dosis nominal igual no garantiza igual número de células viables ni igual estado fisiológico. Variar X₀ no demuestra que la primera muestra estuviera sesgada.
- **Reloj de mayo:** las hojas LAB010–012 usan la primera muestra como t=0, mientras Datos_homologados le asigna t=5 h; el calendario sitúa las siembras alrededor de esa primera muestra. No se cambia aquí el reloj para compensar la discrepancia.
- **Etanol de mayo:** la columna ETANOL existe en las hojas LAB010–012, pero su primera medición no está respaldada por la columna Alcolyzer Real de Resultados_primarios. Esto limita la trazabilidad; no autoriza declarar falsos todos los datos de etanol.
- **Nutrición:** los productos y los registros de tiempos difieren. En mayo la primera muestra figura a las 15:00, antes de la siembra/nutrición inicial de LAB011 y LAB012 según el calendario (15:15 y 15:45), aunque sus eventos no certifican ejecución. Si esa muestra fue anterior a la primera nutrición, su aporte no estaría representado por N0; no puede asumirse sin conciliar el registro. La sensibilidad conserva los pulsos cinéticos históricos de 80 mg/L, incluyendo sus discrepancias frente al calendario; no corrige dosis ni ejecución mediante X₀.
- **Calidad de N:** un aumento entre muestras puede superar lo explicable por las entradas declaradas. Se verifica abajo el balance del modelo para localizar esos intervalos. Es una incompatibilidad entre observaciones y entradas/modelo, no una prueba aislada de error analítico.

No se excluye ningún experimento de la calibración ni se recalibra theta en este análisis. La influencia de retirar datos del ajuste requeriría otra prueba explícita. Los resultados nuevos de esta sección y de la sensibilidad se conservan dentro del EXECUTED, **sin nuevos CSV ni exportaciones de figuras**.

In [ ]:
sx_n_balance_rows = []
for lab in SX_LABS:
    p = observations[observations.batch.eq(lab) & observations.state.eq("N")].sort_values("time_h")
    batch = batch_by_lab[lab]
    for a,b in zip(p.itertuples(), list(p.itertuples())[1:]):
        # Muestra antes de acción: un pulso en a sí entra antes de b; un pulso en b aún no.
        dose = sum(1000*v for t,v in batch.pulses["N"] if t >= a.time_h-1e-8 and t < b.time_h-1e-8)
        rise = b.observed-a.observed
        sx_n_balance_rows.append({"LAB":lab, "t_a_h":a.time_h, "t_b_h":b.time_h,
            "N_a_mg_L":a.observed, "N_b_mg_L":b.observed, "dosis_modelo_intervalo_mg_L":dose,
            "aumento_N_mg_L":rise, "exceso_sobre_entrada_mg_L":rise-dose,
            "incompatible_sin_otra_fuente_segun_modelo":bool(rise-dose > 1e-6)})
sx_n_balance = pd.DataFrame(sx_n_balance_rows)
display(sx_n_balance[sx_n_balance.incompatible_sin_otra_fuente_segun_modelo].round(3))
display(Markdown("El modelo tiene dN/dt ≤ 0 entre pulsos. Cambiar X0 modifica la velocidad de consumo, "
    "pero no puede producir una subida de N sin otra entrada o mecanismo. Los excesos pequeños no se interpretan como significativos sin incertidumbre analítica."))

## 8. Sensibilidad a biomasa viable inicial con theta_full congelado

Se cambia **únicamente X₀**, manteniendo Xd₀, N₀, G₀, F₀, E₀, temperatura, tiempos y pulsos. Esta es una sensibilidad de la entrada inicial del modelo, no una modificación de las mediciones, de la masa por célula ni un ajuste de parámetros. No se alteran todas las observaciones de biomasa mediante un nuevo factor de conversión.

- **LAB006:** X₀ medida × {1, 1,10, 1,25, 1,50, 2}. Los tres primeros aumentos son pruebas moderadas; duplicar es un escenario ampliado explícito.
- **LAB010–012:** X₀ medida × {1, 0,90, 0,75, 0,50, 0,25}. Reducir al 25 % es una prueba ampliada, no una pequeña corrección de medición.

Los factores están definidos antes de simular y se muestran con sus valores absolutos. Theta_defaults con X₀ original se conserva como referencia de contexto; la sensibilidad se calcula con theta_full, que es el vector de interés.

Se evalúan G, F y E como objetivo principal, con **igual peso por variable** después de normalizar por las escalas históricas: raíz de la media de los tres MSE normalizados. N se reporta por separado para ver el coste de adelantar o retrasar su agotamiento. G+F es derivada y no se cuenta dos veces. La biomasa viable se muestra para comprobar las consecuencias sobre su trayectoria; Xd y Gly no participan en este criterio.

Se conservan exactamente las muestras evaluadas y la exclusión de entradas iniciales de la comparación original. En particular, el dato original X₀ sigue visible como círculo abierto aunque el escenario use otro valor, y su diferencia se declara en la tabla. El mínimo entre estos escenarios no estima la biomasa verdadera ni constituye una corrección validada del dataset.

In [ ]:
SX_FACTORS = {"LAB006":(1.,1.10,1.25,1.50,2.),
              "LAB010":(1.,.90,.75,.50,.25),
              "LAB011":(1.,.90,.75,.50,.25),
              "LAB012":(1.,.90,.75,.50,.25)}
SX_STATES = ("G","F","E","N","GF","X")
sx_theta_snapshot = dict(theta_full)
sx_original_x0 = {lab:batch_by_lab[lab].initials["X"] for lab in SX_LABS}
sx_point_frames, sx_curve_frames, sx_scenario_rows = [], [], []
sx_initial_rate_checks = []
for sx_lab,sx_factors in SX_FACTORS.items():
    sx_base = batch_by_lab[sx_lab]
    sx_plot_grid = trajectories[trajectories.theta.eq("theta_full") & trajectories.batch.eq(sx_lab)
        & trajectories.state.eq("X")].time_h.to_numpy(float)
    sx_reference_rate = np.asarray(kinetic.rhs(float(sx_base.time[0]),kinetic.initial_vector(sx_base),theta_full,sx_base))
    for sx_factor in sx_factors:
        sx_x0 = float(sx_base.initials["X"] * sx_factor)
        sx_batch = replace(sx_base, initials={**sx_base.initials,"X":sx_x0})
        assert all(sx_batch.initials[s]==sx_base.initials[s] for s in CORE if s!="X")
        assert sx_batch.pulses == sx_base.pulses
        assert np.array_equal(sx_batch.time,sx_base.time)
        assert np.array_equal(sx_batch.temperature_c,sx_base.temperature_c)
        sx_sample = kinetic.simulate(sx_batch,theta_full,sx_base.time,sample_event_order="sample_before_action")
        sx_dense = kinetic.simulate(sx_batch,theta_full,sx_plot_grid,sample_event_order="sample_before_action")
        assert sx_sample is not None and sx_dense is not None,(sx_lab,sx_factor)
        assert np.isfinite(sx_sample.to_numpy()).all() and np.isfinite(sx_dense.to_numpy()).all()
        sx_sample["GF"] = sx_sample.G+sx_sample.F
        sx_dense["GF"] = sx_dense.G+sx_dense.F
        sx_rates = np.asarray(kinetic.rhs(float(sx_base.time[0]),kinetic.initial_vector(sx_batch),theta_full,sx_batch))
        # A mismo estado químico y T, los flujos iniciales N/G/F/E escalan con X0.
        for sx_state in ("N","G","F","E"):
            sx_i = CORE.index(sx_state)
            assert np.isclose(sx_rates[sx_i],sx_factor*sx_reference_rate[sx_i],rtol=1e-12,atol=1e-12)
        sx_initial_rate_checks.append({"LAB":sx_lab,"factor":sx_factor,"checked_initial_rates":True})
        sx_scenario_rows.append({"LAB":sx_lab,"factor":sx_factor,"X0_medido_g_L":sx_base.initials["X"],
            "X0_escenario_g_L":sx_x0,"delta_X0_g_L":sx_x0-sx_base.initials["X"],
            "cambio_porcentaje":100*(sx_factor-1),"escenario_ampliado":bool(sx_factor in (.25,2.))})
        for sx_state in SX_STATES:
            sx_p = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)].copy()
            sx_p["theta"] = "theta_full"
            sx_p["factor"] = sx_factor
            sx_p["predicted"] = sx_sample.loc[sx_p.time_h.to_numpy(),sx_state].to_numpy()*FACTORS[sx_state]
            sx_p["error"] = sx_p.predicted-sx_p.observed
            sx_point_frames.append(sx_p)
            sx_curve_frames.append(pd.DataFrame({"LAB":sx_lab,"factor":sx_factor,"state":sx_state,
                "time_h":sx_dense.index.to_numpy(),"predicted":sx_dense[sx_state].to_numpy()*FACTORS[sx_state]}))
    print(sx_lab,"— cinco escenarios terminados",flush=True)
sx_points = pd.concat(sx_point_frames,ignore_index=True)
sx_curves = pd.concat(sx_curve_frames,ignore_index=True)
sx_scenarios = pd.DataFrame(sx_scenario_rows)
sx_metrics = summarize_errors(sx_points,["batch","factor","state"])
sx_metrics["MSE_historical"] = sx_metrics.WRMSE_historical**2
sx_metric_wide = sx_metrics.pivot(index=["batch","factor"],columns="state",values="RMSE")
sx_summary_rows = []
for sx_row in sx_scenarios.itertuples():
    sx_m = sx_metrics[sx_metrics.batch.eq(sx_row.LAB) & sx_metrics.factor.eq(sx_row.factor)].set_index("state")
    sx_gfe = sx_points[sx_points.batch.eq(sx_row.LAB) & sx_points.factor.eq(sx_row.factor)
        & sx_points.state.isin(("G","F","E")) & sx_points.used_in_primary_score]
    sx_np = sx_points[sx_points.batch.eq(sx_row.LAB) & sx_points.factor.eq(sx_row.factor)
        & sx_points.state.eq("N") & sx_points.used_in_primary_score]
    sx_pulse_h = min(t for t,_ in batch_by_lab[sx_row.LAB].pulses["N"])
    sx_pre = sx_np[sx_np.time_h.le(sx_pulse_h+1e-8)]
    sx_nc = sx_curves[sx_curves.LAB.eq(sx_row.LAB) & sx_curves.factor.eq(sx_row.factor)
        & sx_curves.state.eq("N") & sx_curves.time_h.le(sx_pulse_h)]
    sx_cross = sx_nc[sx_nc.predicted.le(5)]
    sx_summary_rows.append({"LAB":sx_row.LAB,"factor":sx_row.factor,"X0_g_L":sx_row.X0_escenario_g_L,
        "cambio_X0_pct":sx_row.cambio_porcentaje,"escenario_ampliado":sx_row.escenario_ampliado,
        "score_GFE_igual_peso":float(np.sqrt(sx_m.loc[["G","F","E"],"MSE_historical"].mean())),
        "score_GFE_por_observacion":float(np.sqrt(np.mean((sx_gfe.error/sx_gfe.sigma_historical)**2))),
        **{f"RMSE_{s}":sx_m.loc[s,"RMSE"] for s in SX_STATES},
        "RMSE_N_prepulso_mg_L":float(np.sqrt(np.mean(sx_pre.error**2))) if len(sx_pre) else np.nan,
        "primer_N_le_5_mg_L_prepulso_h":float(sx_cross.time_h.iloc[0]) if len(sx_cross) else np.nan,
        "n_G":int(sx_m.loc["G","n"]),"n_F":int(sx_m.loc["F","n"]),"n_E":int(sx_m.loc["E","n"]),"n_N":int(sx_m.loc["N","n"])})
sx_summary = pd.DataFrame(sx_summary_rows)
sx_baseline = sx_summary[sx_summary.factor.eq(1.)].set_index("LAB")
sx_summary["delta_score_GFE_pct"] = [100*(r.score_GFE_igual_peso/sx_baseline.loc[r.LAB,"score_GFE_igual_peso"]-1) for r in sx_summary.itertuples()]
sx_summary["delta_RMSE_N_pct"] = [100*(r.RMSE_N/sx_baseline.loc[r.LAB,"RMSE_N"]-1) for r in sx_summary.itertuples()]
sx_summary["delta_RMSE_X_pct"] = [100*(r.RMSE_X/sx_baseline.loc[r.LAB,"RMSE_X"]-1) for r in sx_summary.itertuples()]
display(sx_scenarios.round(5))
display(sx_summary[["LAB","factor","X0_g_L","score_GFE_igual_peso","delta_score_GFE_pct","RMSE_G","RMSE_F","RMSE_E","RMSE_N","delta_RMSE_N_pct","RMSE_X"]].round(4))
display(sx_summary[["LAB","factor","n_G","n_F","n_E","n_N","RMSE_N_prepulso_mg_L","primer_N_le_5_mg_L_prepulso_h"]].round(3))

### 8.1 Curvas con X₀ alternativo

Cada figura muestra theta_full con cinco valores de X₀, junto a theta_defaults original en gris discontinuo. Los puntos son las mismas observaciones del análisis inicial; los círculos abiertos siguen señalando las entradas originales. Las líneas rosadas son los pulsos del modelo. El eje de N mantiene el pequeño margen inferior para que las curvas en cero se vean completas.

In [ ]:
sx_lab = 'LAB006'
sx_plot_states = ("G","F","E","N","GF","X")
sx_colors = {factor:color for factor,color in zip(SX_FACTORS[sx_lab],["#222222","#0072B2","#009E73","#E69F00","#CC79A7"])}
sx_fig,sx_axes = plt.subplots(2,3,figsize=(16,9),layout="constrained")
for sx_ax,sx_state in zip(sx_axes.flat,sx_plot_states):
    for sx_factor in SX_FACTORS[sx_lab]:
        sx_c = sx_curves[sx_curves.LAB.eq(sx_lab) & sx_curves.factor.eq(sx_factor) & sx_curves.state.eq(sx_state)]
        sx_x0 = sx_original_x0[sx_lab]*sx_factor
        sx_label = f"X0 ×{sx_factor:g} = {sx_x0:.4f} g/L" + (" (original)" if sx_factor==1 else "")
        sx_ax.plot(sx_c.time_h,sx_c.predicted,color=sx_colors[sx_factor],lw=2.2 if sx_factor==1 else 1.7,label=sx_label)
    sx_context = trajectories[trajectories.theta.eq("theta_defaults") & trajectories.batch.eq(sx_lab) & trajectories.state.eq(sx_state)]
    sx_ax.plot(sx_context.time_h,sx_context.predicted,color="#888888",lw=1.2,ls="--",label="theta_defaults · X0 original")
    sx_p = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)]
    sx_ax.scatter(sx_p.loc[~sx_p.initial_input_anchor,"time_h"],sx_p.loc[~sx_p.initial_input_anchor,"observed"],
        color="black",edgecolors="white",linewidths=.4,s=30,zorder=6,label="Muestras reales")
    sx_ax.scatter(sx_p.loc[sx_p.initial_input_anchor,"time_h"],sx_p.loc[sx_p.initial_input_anchor,"observed"],
        facecolors="white",edgecolors="black",s=38,zorder=7,label="Entrada inicial medida")
    for sx_t,_ in batch_by_lab[sx_lab].pulses["N"]:
        sx_ax.axvline(sx_t,color="#CC79A7",ls=":",lw=1,alpha=.8)
    sx_ax.set(title=LABELS[sx_state],xlabel="Tiempo histórico [h]",ylabel=f"{sx_state} [{UNITS[sx_state]}]",
        xlim=(0,batch_by_lab[sx_lab].time[-1]))
    sx_ax.set_ylim(bottom=-.02*sx_ax.get_ylim()[1] if sx_state=="N" else 0)
sx_fig.get_layout_engine().set(rect=(0,0,1,.81))
sx_handles,sx_labels = sx_axes.flat[0].get_legend_handles_labels()
sx_fig.legend(sx_handles,sx_labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=3,fontsize=9)
sx_fig.suptitle(f"{sx_lab} · sensibilidad a biomasa viable inicial · theta_full congelado",y=.995,fontsize=14)
plt.show()
plt.close(sx_fig)

In [ ]:
sx_lab = 'LAB010'
sx_plot_states = ("G","F","E","N","GF","X")
sx_colors = {factor:color for factor,color in zip(SX_FACTORS[sx_lab],["#222222","#0072B2","#009E73","#E69F00","#CC79A7"])}
sx_fig,sx_axes = plt.subplots(2,3,figsize=(16,9),layout="constrained")
for sx_ax,sx_state in zip(sx_axes.flat,sx_plot_states):
    for sx_factor in SX_FACTORS[sx_lab]:
        sx_c = sx_curves[sx_curves.LAB.eq(sx_lab) & sx_curves.factor.eq(sx_factor) & sx_curves.state.eq(sx_state)]
        sx_x0 = sx_original_x0[sx_lab]*sx_factor
        sx_label = f"X0 ×{sx_factor:g} = {sx_x0:.4f} g/L" + (" (original)" if sx_factor==1 else "")
        sx_ax.plot(sx_c.time_h,sx_c.predicted,color=sx_colors[sx_factor],lw=2.2 if sx_factor==1 else 1.7,label=sx_label)
    sx_context = trajectories[trajectories.theta.eq("theta_defaults") & trajectories.batch.eq(sx_lab) & trajectories.state.eq(sx_state)]
    sx_ax.plot(sx_context.time_h,sx_context.predicted,color="#888888",lw=1.2,ls="--",label="theta_defaults · X0 original")
    sx_p = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)]
    sx_ax.scatter(sx_p.loc[~sx_p.initial_input_anchor,"time_h"],sx_p.loc[~sx_p.initial_input_anchor,"observed"],
        color="black",edgecolors="white",linewidths=.4,s=30,zorder=6,label="Muestras reales")
    sx_ax.scatter(sx_p.loc[sx_p.initial_input_anchor,"time_h"],sx_p.loc[sx_p.initial_input_anchor,"observed"],
        facecolors="white",edgecolors="black",s=38,zorder=7,label="Entrada inicial medida")
    for sx_t,_ in batch_by_lab[sx_lab].pulses["N"]:
        sx_ax.axvline(sx_t,color="#CC79A7",ls=":",lw=1,alpha=.8)
    sx_ax.set(title=LABELS[sx_state],xlabel="Tiempo histórico [h]",ylabel=f"{sx_state} [{UNITS[sx_state]}]",
        xlim=(0,batch_by_lab[sx_lab].time[-1]))
    sx_ax.set_ylim(bottom=-.02*sx_ax.get_ylim()[1] if sx_state=="N" else 0)
sx_fig.get_layout_engine().set(rect=(0,0,1,.81))
sx_handles,sx_labels = sx_axes.flat[0].get_legend_handles_labels()
sx_fig.legend(sx_handles,sx_labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=3,fontsize=9)
sx_fig.suptitle(f"{sx_lab} · sensibilidad a biomasa viable inicial · theta_full congelado",y=.995,fontsize=14)
plt.show()
plt.close(sx_fig)

In [ ]:
sx_lab = 'LAB011'
sx_plot_states = ("G","F","E","N","GF","X")
sx_colors = {factor:color for factor,color in zip(SX_FACTORS[sx_lab],["#222222","#0072B2","#009E73","#E69F00","#CC79A7"])}
sx_fig,sx_axes = plt.subplots(2,3,figsize=(16,9),layout="constrained")
for sx_ax,sx_state in zip(sx_axes.flat,sx_plot_states):
    for sx_factor in SX_FACTORS[sx_lab]:
        sx_c = sx_curves[sx_curves.LAB.eq(sx_lab) & sx_curves.factor.eq(sx_factor) & sx_curves.state.eq(sx_state)]
        sx_x0 = sx_original_x0[sx_lab]*sx_factor
        sx_label = f"X0 ×{sx_factor:g} = {sx_x0:.4f} g/L" + (" (original)" if sx_factor==1 else "")
        sx_ax.plot(sx_c.time_h,sx_c.predicted,color=sx_colors[sx_factor],lw=2.2 if sx_factor==1 else 1.7,label=sx_label)
    sx_context = trajectories[trajectories.theta.eq("theta_defaults") & trajectories.batch.eq(sx_lab) & trajectories.state.eq(sx_state)]
    sx_ax.plot(sx_context.time_h,sx_context.predicted,color="#888888",lw=1.2,ls="--",label="theta_defaults · X0 original")
    sx_p = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)]
    sx_ax.scatter(sx_p.loc[~sx_p.initial_input_anchor,"time_h"],sx_p.loc[~sx_p.initial_input_anchor,"observed"],
        color="black",edgecolors="white",linewidths=.4,s=30,zorder=6,label="Muestras reales")
    sx_ax.scatter(sx_p.loc[sx_p.initial_input_anchor,"time_h"],sx_p.loc[sx_p.initial_input_anchor,"observed"],
        facecolors="white",edgecolors="black",s=38,zorder=7,label="Entrada inicial medida")
    for sx_t,_ in batch_by_lab[sx_lab].pulses["N"]:
        sx_ax.axvline(sx_t,color="#CC79A7",ls=":",lw=1,alpha=.8)
    sx_ax.set(title=LABELS[sx_state],xlabel="Tiempo histórico [h]",ylabel=f"{sx_state} [{UNITS[sx_state]}]",
        xlim=(0,batch_by_lab[sx_lab].time[-1]))
    sx_ax.set_ylim(bottom=-.02*sx_ax.get_ylim()[1] if sx_state=="N" else 0)
sx_fig.get_layout_engine().set(rect=(0,0,1,.81))
sx_handles,sx_labels = sx_axes.flat[0].get_legend_handles_labels()
sx_fig.legend(sx_handles,sx_labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=3,fontsize=9)
sx_fig.suptitle(f"{sx_lab} · sensibilidad a biomasa viable inicial · theta_full congelado",y=.995,fontsize=14)
plt.show()
plt.close(sx_fig)

In [ ]:
sx_lab = 'LAB012'
sx_plot_states = ("G","F","E","N","GF","X")
sx_colors = {factor:color for factor,color in zip(SX_FACTORS[sx_lab],["#222222","#0072B2","#009E73","#E69F00","#CC79A7"])}
sx_fig,sx_axes = plt.subplots(2,3,figsize=(16,9),layout="constrained")
for sx_ax,sx_state in zip(sx_axes.flat,sx_plot_states):
    for sx_factor in SX_FACTORS[sx_lab]:
        sx_c = sx_curves[sx_curves.LAB.eq(sx_lab) & sx_curves.factor.eq(sx_factor) & sx_curves.state.eq(sx_state)]
        sx_x0 = sx_original_x0[sx_lab]*sx_factor
        sx_label = f"X0 ×{sx_factor:g} = {sx_x0:.4f} g/L" + (" (original)" if sx_factor==1 else "")
        sx_ax.plot(sx_c.time_h,sx_c.predicted,color=sx_colors[sx_factor],lw=2.2 if sx_factor==1 else 1.7,label=sx_label)
    sx_context = trajectories[trajectories.theta.eq("theta_defaults") & trajectories.batch.eq(sx_lab) & trajectories.state.eq(sx_state)]
    sx_ax.plot(sx_context.time_h,sx_context.predicted,color="#888888",lw=1.2,ls="--",label="theta_defaults · X0 original")
    sx_p = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)]
    sx_ax.scatter(sx_p.loc[~sx_p.initial_input_anchor,"time_h"],sx_p.loc[~sx_p.initial_input_anchor,"observed"],
        color="black",edgecolors="white",linewidths=.4,s=30,zorder=6,label="Muestras reales")
    sx_ax.scatter(sx_p.loc[sx_p.initial_input_anchor,"time_h"],sx_p.loc[sx_p.initial_input_anchor,"observed"],
        facecolors="white",edgecolors="black",s=38,zorder=7,label="Entrada inicial medida")
    for sx_t,_ in batch_by_lab[sx_lab].pulses["N"]:
        sx_ax.axvline(sx_t,color="#CC79A7",ls=":",lw=1,alpha=.8)
    sx_ax.set(title=LABELS[sx_state],xlabel="Tiempo histórico [h]",ylabel=f"{sx_state} [{UNITS[sx_state]}]",
        xlim=(0,batch_by_lab[sx_lab].time[-1]))
    sx_ax.set_ylim(bottom=-.02*sx_ax.get_ylim()[1] if sx_state=="N" else 0)
sx_fig.get_layout_engine().set(rect=(0,0,1,.81))
sx_handles,sx_labels = sx_axes.flat[0].get_legend_handles_labels()
sx_fig.legend(sx_handles,sx_labels,loc="upper center",bbox_to_anchor=(.5,.965),ncol=3,fontsize=9)
sx_fig.suptitle(f"{sx_lab} · sensibilidad a biomasa viable inicial · theta_full congelado",y=.995,fontsize=14)
plt.show()
plt.close(sx_fig)

### 8.2 Compromiso entre ajuste de azúcares/etanol y nitrógeno

En cada panel, desplazarse a la izquierda reduce el error conjunto normalizado de G/F/E; bajar reduce el RMSE de N. La estrella marca la entrada original. Un escenario que mejore G/F/E puede empeorar N. No se impone una combinación arbitraria de ambos objetivos ni se declara un X₀ óptimo a partir de la figura.

In [ ]:
sx_fig,sx_axes = plt.subplots(2,2,figsize=(12,8),layout="constrained")
for sx_ax,sx_lab in zip(sx_axes.flat,SX_LABS):
    sx_sub = sx_summary[sx_summary.LAB.eq(sx_lab)]
    sx_colors = dict(zip(SX_FACTORS[sx_lab],["#222222","#0072B2","#009E73","#E69F00","#CC79A7"]))
    for sx_i,sx_r in enumerate(sx_sub.itertuples()):
        sx_ax.scatter(sx_r.score_GFE_igual_peso,sx_r.RMSE_N,color=sx_colors[sx_r.factor],
            s=160 if sx_r.factor==1 else 55,marker="*" if sx_r.factor==1 else "o",zorder=4)
        sx_ax.annotate(f"×{sx_r.factor:g}",(sx_r.score_GFE_igual_peso,sx_r.RMSE_N),
            xytext=(5,7 if sx_i%2==0 else -13),textcoords="offset points",fontsize=9)
    sx_ax.margins(x=.18,y=.20)
    # No amplificar diferencias de redondeo cuando el error de N es constante.
    sx_min_span = max(1., .04*abs(float(sx_sub.RMSE_N.mean())))
    if np.ptp(sx_sub.RMSE_N.to_numpy()) < sx_min_span:
        sx_center = float(sx_sub.RMSE_N.mean())
        sx_ax.set_ylim(sx_center-.5*sx_min_span,sx_center+.5*sx_min_span)
    sx_ax.ticklabel_format(axis="y",style="plain",useOffset=False)
    sx_ax.set(title=sx_lab,xlabel="Error G/F/E normalizado · igual peso por variable",ylabel="RMSE N [mg N/L]")
sx_fig.suptitle("Sensibilidad a X0 · estrella: entrada original · izquierda y abajo: menor error")
plt.show()
plt.close(sx_fig)

## 9. Qué responde la sensibilidad y qué queda pendiente

Se resume tanto el menor error G/F/E de la malla moderada (factores 0,5–1,5) como el menor de toda la malla, incluidos los extremos exploratorios. Esta selección describe los escenarios ensayados; no es una estimación de X₀, no certifica un sesgo en las muestras y no reemplaza el vector de calibración.

In [ ]:
sx_best_rows = []
for sx_lab in SX_LABS:
    sx_sub = sx_summary[sx_summary.LAB.eq(sx_lab)]
    for sx_scope,sx_candidates in [("moderada (0.5–1.5)",sx_sub[sx_sub.factor.between(.5,1.5)]),("completa",sx_sub)]:
        sx_best = sx_candidates.loc[sx_candidates.score_GFE_igual_peso.idxmin()]
        sx_best_rows.append({"LAB":sx_lab,"malla":sx_scope,"factor_min_GFE":float(sx_best.factor),
            "X0_g_L":float(sx_best.X0_g_L),"delta_score_GFE_pct":float(sx_best.delta_score_GFE_pct),
            "RMSE_N_mg_L":float(sx_best.RMSE_N),"delta_RMSE_N_pct":float(sx_best.delta_RMSE_N_pct),
            "RMSE_X_g_L":float(sx_best.RMSE_X),"escenario_ampliado":bool(sx_best.escenario_ampliado)})
sx_best_explored = pd.DataFrame(sx_best_rows)
display(sx_best_explored.round(4))
sx_lines = []
for sx_lab in SX_LABS:
    sx_base = sx_baseline.loc[sx_lab]
    sx_moderate = sx_best_explored[sx_best_explored.LAB.eq(sx_lab) & sx_best_explored.malla.str.startswith("moderada")].iloc[0]
    sx_complete = sx_best_explored[sx_best_explored.LAB.eq(sx_lab) & sx_best_explored.malla.eq("completa")].iloc[0]
    sx_lines.append(f"- **{sx_lab}:** X0 original **{sx_original_x0[sx_lab]:.4f} g/L**. En la malla moderada, "
        f"el menor error G/F/E aparece con **×{sx_moderate.factor_min_GFE:g}**: cambio de error "
        f"**{sx_moderate.delta_score_GFE_pct:+.1f}%** y de RMSE de N **{sx_moderate.delta_RMSE_N_pct:+.1f}%**. "
        f"En la malla completa, el mínimo G/F/E está en **×{sx_complete.factor_min_GFE:g}** "
        f"(**{sx_complete.X0_g_L:.4f} g/L**, cambio **{sx_complete.delta_score_GFE_pct:+.1f}%**).")
sx_conflicts = sx_n_balance[sx_n_balance.exceso_sobre_entrada_mg_L.gt(20)]
sx_lines += [
    "- Los programas térmicos C/D/E/F y el cambio de producto nutricional están documentados. La dosis registrada de levadura es 1 g en 2 L en todos estos casos; no se halló una explicación explícita de los recuentos iniciales extremos en las fuentes revisadas.",
    f"- Hay **{len(sx_conflicts)} intervalo(s)** de los cuatro LAB con una subida de N que supera las entradas cinéticas declaradas en más de 20 mg/L. "
    "Ese umbral es descriptivo, no un criterio analítico certificado. Ningún cambio de X0 puede generar esas subidas con las ecuaciones actuales.",
    "- La muestra inicial sigue siendo una observación original, incluso cuando el escenario modifica X0. La diferencia puede representar incertidumbre de muestreo, conversión recuento/biomasa o fracción activa, pero estos mecanismos no son identificables por esta sensibilidad aislada.",
    "- Las diferencias de reloj, registros planificados frente a ejecución y trazabilidad del etanol de mayo impiden concluir que X0 sea la única causa. Tampoco demuestran que toda la química sea mala: las discrepancias deben resolverse por variable y fuente.",
    "- No se eliminó LAB006 ni LAB010–012 de ninguna calibración. No se recalibraron theta, tiempos, temperatura ni pulsos. Para evaluar cuánto contaminan el ajuste habría que contrastar después calibraciones con conjuntos de datos conciliados y una evaluación independiente.",
]
for sx_factor in (1.10,1.25):
    sx_small = sx_summary[sx_summary.LAB.eq("LAB006") & sx_summary.factor.eq(sx_factor)].iloc[0]
    sx_lines.append(f"- **LAB006, cambio pequeño ×{sx_factor:g}:** error G/F/E "
        f"**{sx_small.delta_score_GFE_pct:+.1f}%** y RMSE N **{sx_small.delta_RMSE_N_pct:+.1f}%** respecto del original.")
sx_lines.append("- En los cuatro LAB, el menor error G/F/E de la malla completa cae en el extremo explorado (×2 o ×0.25). "
    "No se ha localizado un mínimo interior que identifique una biomasa inicial óptima.")
sx_lines.append("- **LAB012:** N sube de 222.06 a 277.52 mg/L entre 0 y 48 h, antes del pulso dinámico usado por el modelo. "
    "La muestra inicial está etiquetada preinóculo y el calendario programa una nutrición inicial de 80 ppm YAN después de ella. "
    "Si ese orden se ejecutó realmente, una entrada inicial no contabilizada podría permitir la subida de 55.46 mg/L; "
    "esta es una hipótesis de conciliación del registro, no una corrección aplicada ni una prueba de dato analítico malo.")
display(Markdown("\n".join(sx_lines)))
# Verificación: mismos datos evaluados, baseline reproducida y solo X0 cambia.
assert theta_full == sx_theta_snapshot
assert all(batch_by_lab[lab].initials["X"]==value for lab,value in sx_original_x0.items())
assert len(sx_scenarios)==20 and len(sx_metrics)==20*len(SX_STATES)
for sx_lab in SX_LABS:
    for sx_state in SX_STATES:
        sx_reference = points[points.theta.eq("theta_full") & points.batch.eq(sx_lab) & points.state.eq(sx_state)].sort_values("time_h")
        for sx_factor in SX_FACTORS[sx_lab]:
            sx_test = sx_points[sx_points.batch.eq(sx_lab) & sx_points.factor.eq(sx_factor) & sx_points.state.eq(sx_state)].sort_values("time_h")
            assert np.array_equal(sx_reference.time_h.to_numpy(),sx_test.time_h.to_numpy())
            assert np.array_equal(sx_reference.observed.to_numpy(),sx_test.observed.to_numpy())
            assert np.array_equal(sx_reference.used_in_primary_score.to_numpy(),sx_test.used_in_primary_score.to_numpy())
            if sx_factor==1:
                assert np.allclose(sx_reference.predicted,sx_test.predicted,rtol=1e-10,atol=1e-10)
for sx_row in sx_metrics.itertuples():
    sx_test = sx_points[sx_points.batch.eq(sx_row.batch) & sx_points.factor.eq(sx_row.factor)
        & sx_points.state.eq(sx_row.state) & sx_points.used_in_primary_score]
    assert np.isclose(sx_row.RMSE**2,np.mean((sx_test.predicted-sx_test.observed)**2),rtol=1e-12)
assert INPUT_HASHES == {k:file_hash(p) for k,p in INPUTS.items()}
print("Sensibilidad verificada: 4 LAB, 20 escenarios, seis variables; sin nuevas exportaciones CSV ni ajustes de parámetros.")
# Solo se amplía el manifiesto existente; las nuevas tablas permanecen inline.
manifest["X0_sensitivity"] = {
    "labs":list(SX_LABS), "factors":{lab:list(factors) for lab,factors in SX_FACTORS.items()},
    "theta":"theta_full frozen", "changed_input":"viable biomass X0 only",
    "primary_states":["G","F","E"], "primary_weighting":"equal weight per variable using historical sigmas",
    "secondary_states":["N","GF","X"], "new_results_location":"EXECUTED sections 7–9; no new CSV files",
    "original_conclusions_json_scope":"baseline comparison, section 6",
    "protocol_review":"source workbook and calendar read-only; planned events do not certify execution",
}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")